# Assignmnet 2 (100 points)

**Name:** ADIL AHMED <br>

### SMS Spam Detection *(60 points)*

<p>You are hired as an AI expert in the development department of a telecommunications company. The first thing on your orientation plan is a small project that your boss has assigned you for the following given situation. Your supervisor has given away his private cell phone number on too many websites and is now complaining about daily spam SMS. Therefore, it is your job to write a spam detector in Python. </p>

<p>In doing so, you need to use a Naive Bayes classifier that can handle both bag-of-words (BoW) and tf-idf features as input. For the evaluation of your spam detector, an SMS collection is available as a dataset - this has yet to be suitably split into train and test data. To keep the costs as low as possible and to avoid problems with copyrights, your boss insists on a new development with Python.</p>

<p>Include a short description of the data preprocessing steps, method, experiment design, hyper-parameters, and evaluation metric. Also, document your findings, drawbacks, and potential improvements.</p>

<p>Note: You need to implement the bag-of-words (BoW) and tf-idf feature extractor from scratch. You can use existing python libraries for other tasks.</p>

**Dataset and Resources**

* SMS Spam Collection Dataset: https://archive.ics.uci.edu/dataset/228/sms+spam+collection

In [3]:
import math
import re
from collections import Counter, defaultdict
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

def load_data(filepath):
    labels = []
    texts = []
    with open(filepath, "r", encoding="utf-8") as f:
        lines = f.readlines()
        for line in lines:
            if '\t' in line:
                label, text = line.strip().split("\t", 1)
                labels.append(1 if label.strip().lower() == 'spam' else 0)
                texts.append(text.strip().lower())
    return texts, labels

def tokenize(text):
    return re.findall(r'\b\w+\b', text)

def bag_of_words(docs, vocabulary=None):
    build_vocab = vocabulary is None
    if build_vocab:
        vocabulary = sorted(set(word for doc in docs for word in tokenize(doc)))
    word_to_index = {word: i for i, word in enumerate(vocabulary)}
    features = []
    for doc in docs:
        vec = [0] * len(vocabulary)
        for word in tokenize(doc):
            if word in word_to_index:
                vec[word_to_index[word]] += 1
        features.append(vec)
    return (features, vocabulary) if build_vocab else features

def tf_idf(docs, vocabulary=None, df=None):
    build_vocab = vocabulary is None or df is None
    if build_vocab:
        vocabulary = sorted(set(word for doc in docs for word in tokenize(doc)))
        word_to_index = {word: i for i, word in enumerate(vocabulary)}
        df = defaultdict(int)
        for doc in docs:
            for word in set(tokenize(doc)):
                df[word] += 1
    else:
        word_to_index = {word: i for i, word in enumerate(vocabulary)}
    doc_count = len(docs)
    features = []
    for doc in docs:
        tf = Counter(tokenize(doc))
        vec = [0.0] * len(vocabulary)
        for word in tf:
            if word in word_to_index:
                i = word_to_index[word]
                idf = math.log((doc_count + 1) / (df.get(word, 0) + 1)) + 1
                vec[i] = tf[word] * idf
        features.append(vec)
    return (features, vocabulary, df) if build_vocab else features

if __name__ == "__main__":
    data_filepath = "sms_spam_collection/SMSSpamCollection"
    texts, labels = load_data(data_filepath)
    X_train_texts, X_test_texts, y_train, y_test = train_test_split(
        texts, labels, test_size=0.2, random_state=42, stratify=labels
    )

    # Bag of Words
    X_train_bow, vocab_bow = bag_of_words(X_train_texts)
    X_test_bow = bag_of_words(X_test_texts, vocabulary=vocab_bow)
    clf_bow = MultinomialNB().fit(X_train_bow, y_train)
    preds_bow = clf_bow.predict(X_test_bow)
    print("\n=== Bag-of-Words ===")
    print(classification_report(y_test, preds_bow, target_names=["ham", "spam"]))

    # TF-IDF
    X_train_tfidf, vocab_tfidf, df_train = tf_idf(X_train_texts)
    X_test_tfidf = tf_idf(X_test_texts, vocabulary=vocab_tfidf, df=df_train)
    clf_tfidf = MultinomialNB().fit(X_train_tfidf, y_train)
    preds_tfidf = clf_tfidf.predict(X_test_tfidf)
    print("\n=== TF-IDF ===")
    print(classification_report(y_test, preds_tfidf, target_names=["ham", "spam"]))


=== Bag-of-Words ===
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       966
        spam       0.98      0.90      0.94       149

    accuracy                           0.98      1115
   macro avg       0.98      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115


=== TF-IDF ===
              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       966
        spam       0.93      0.95      0.94       149

    accuracy                           0.98      1115
   macro avg       0.96      0.97      0.97      1115
weighted avg       0.98      0.98      0.98      1115



 ### Search Engine *(40 points)*

Your boss is impressed with your spam detector and assigns you a new task. As part of improving internal tools, the company wants a search engine that can search through SMS messages and rank them by relevance. Implement the PageRank algorithm from scratch to score each SMS message based on its importance in the document graph.

*   Compute TF-IDF vectors for all SMS messages (you can leverage previous implementation)
*   Construct a document graph, where each node represents an SMS message and edges are the links between nodes.
*  Implement the PageRank algorithm from scratch to assign an importance score to each SMS message based on its position in the document graph.

#### Hint : You can use the previous dataset or any dataset from your choice.




## You might need the follwoing formulas for your implementation

---

### 1) Cosine Similarity Between Two Document Vectors

Cosine similarity measures how similar two vectors are based on the angle between them:

$$
\text{cosine\_sim}(A, B) = \frac{A \cdot B}{\|A\| \cdot \|B\|}
$$

- \( A \cdot B \): Dot product of vectors \( A \) and \( B \)  
- \( \|A\| \): Euclidean norm (magnitude) of vector \( A \)  
- \( \|B\| \): Euclidean norm of vector \( B \)

**Use case**: Comparing TF-IDF vectors to measure similarity between two messages.

---

### 2) PageRank of a Node \( i \)

PageRank estimates the importance of a document based on its connections in a graph:

$$
PR(i) = \frac{1 - d}{N} + d \sum_{j \in M(i)} \frac{PR(j)}{L(j)}
$$

Where:
- \( PR(i) \): PageRank score of node \( i \)  
- \( d \): Damping factor (typically 0.85)  
- \( N \): Total number of nodes (documents) in the graph  
- \( M(i) \): Set of nodes that link to node \( i \)  
- \( L(j) \): Number of outbound links from node \( j \)  

**Interpretation**:  
- A document is important if **important documents link to it**.  
- The score is split among a node’s outbound links.  
- The **teleportation term** $\text(\frac{1 - d}{N})$ accounts for random jumps, ensuring stability and fairness.

---



In [5]:
import numpy as np

def build_adjacency_matrix(tfidf_vectors, threshold=0.2):
    X = np.array(tfidf_vectors)

    dot_product = np.dot(X, X.T)
    norms = np.linalg.norm(X, axis=1)
    norm_matrix = np.outer(norms, norms)
    norm_matrix[norm_matrix == 0] = 1e-10

    cosine_sim_matrix = dot_product / norm_matrix
    np.fill_diagonal(cosine_sim_matrix, 0)

    adjacency_matrix = np.where(cosine_sim_matrix >= threshold, cosine_sim_matrix, 0)
    return adjacency_matrix

def pagerank_from_formula(adj_matrix, damping=0.85, max_iter=30, tol=1e-4):
    N = adj_matrix.shape[0]
    pr = np.ones(N) / N
    out_links = np.count_nonzero(adj_matrix, axis=1)

    for _ in range(max_iter):
        new_pr = np.ones(N) * (1 - damping) / N
        for i in range(N):
            for j in range(N):
                if adj_matrix[j][i]:
                    if out_links[j] == 0:
                        new_pr[i] += damping * (pr[j] / N)
                    else:
                        new_pr[i] += damping * (pr[j] / out_links[j])
        if np.linalg.norm(new_pr - pr, ord=1) < tol:
            break
        pr = new_pr
    return pr

texts, labels = load_data(data_filepath)
tfidf_vectors, vocab, df = tf_idf(texts)

adj_matrix = build_adjacency_matrix(tfidf_vectors, threshold=0.2)
pr_scores = pagerank_from_formula(adj_matrix)

for idx in range(len(texts)):
    print(f"Score: {pr_scores[idx]:.4f} | Message: {texts[idx]}")

Score: 0.0000 | Message: go until jurong point, crazy.. available only in bugis n great world la e buffet... cine there got amore wat...
Score: 0.0002 | Message: ok lar... joking wif u oni...
Score: 0.0001 | Message: free entry in 2 a wkly comp to win fa cup final tkts 21st may 2005. text fa to 87121 to receive entry question(std txt rate)t&c's apply 08452810075over18's
Score: 0.0002 | Message: u dun say so early hor... u c already then say...
Score: 0.0001 | Message: nah i don't think he goes to usf, he lives around here though
Score: 0.0000 | Message: freemsg hey there darling it's been 3 week's now and no word back! i'd like some fun you up for it still? tb ok! xxx std chgs to send, £1.50 to rcv
Score: 0.0001 | Message: even my brother is not like to speak with me. they treat me like aids patent.
Score: 0.0001 | Message: as per your request 'melle melle (oru minnaminunginte nurungu vettam)' has been set as your callertune for all callers. press *9 to copy your friends callertune
Sco

### Additional Experiments *(5 additional points - <span style="color: red;">Optional</span>)*